# 노트북으로 엑셀 · Access 데이터 고치기

화면 프로그램(`data_editor.py`)이 속에서 하는 일을 **한 칸씩 실행하며** 따라가 봅니다.
칸(셀)을 고르고 `Shift + Enter` 를 누르면 그 칸이 실행되고 아래에 결과가 나옵니다. 위에서부터 순서대로 실행하세요.

- 준비(가상 환경 · 라이브러리 설치 · 커널 고르기)는 README 의 「노트북으로 실행하기」를 먼저 끝내야 합니다.
- 여기서 고치는 것은 `samples` 폴더의 **연습용 가상 데이터**입니다. 2번 칸을 다시 실행하면 처음 상태로 돌아갑니다.

## 1. 환경 확인

`실행 위치`가 이 폴더의 `.venv` 안이면 가상 환경이 제대로 골라진 것입니다.

In [1]:
import sys
import openpyxl
import pandas as pd

print("파이썬   :", sys.version.split()[0])
print("실행 위치:", sys.executable)
print("openpyxl :", openpyxl.__version__)
print("pandas   :", pd.__version__)

파이썬   : 3.13.2
실행 위치: C:\Users\사용자\work-data-editor\.venv\Scripts\python.exe
openpyxl : 3.1.5
pandas   : 3.1.1


## 2. 연습용 데이터 준비

`samples_original` 폴더의 원본을 `samples` 폴더로 복사합니다. 연습을 처음부터 다시 하고 싶을 때도 이 칸을 실행합니다.

In [2]:
import runpy

runpy.run_path("reset_samples.py");

준비: 2026_10_부산지사.xlsx
준비: S1_보금자리론_취합.xlsx
준비: 보금자리론_신청_2026_10.xlsx
준비: 업무DB.accdb
준비: 지점코드표.xlsx
끝. (백업 폴더는 그대로 둡니다)


## 3. 엑셀 파일 열기

`open_source` 가 파일 종류(엑셀 / Access)를 알아서 고릅니다. `load` 는 시트 한 장을 메모리로 읽어 옵니다.

In [3]:
import data_editor as ed

src = ed.open_source("samples/보금자리론_신청_2026_10.xlsx")
print("시트 목록:", src.tables())

s = src.load("신청내역")
print("행 수   :", len(s.rows))
print("열과 종류:", [(c.name, c.kind) for c in s.columns])

시트 목록: ['신청내역']
행 수   : 240
열과 종류: [('신청번호', 'str'), ('신청일', 'date'), ('지점코드', 'str'), ('상품명', 'str'), ('신청금액(만원)', 'int'), ('대출기간(년)', 'int'), ('고객구분', 'str'), ('처리상태', 'str')]


표를 보기 좋게 출력하는 작은 함수를 하나 만들어 둡니다. 맨 왼쪽 숫자는 행 번호입니다.

In [4]:
def show(session, rids=None, n=5):
    """행 번호 목록(rids)의 앞 n 줄을 표로 보여 준다."""
    rids = session.ids() if rids is None else list(rids)
    rids = rids[:n]
    return pd.DataFrame([session.rows[r] for r in rids], columns=session.names, index=rids)


show(s)

,신청번호,신청일,지점코드,상품명,신청금액(만원),대출기간(년),고객구분,처리상태
0,HF202610-00505,2026-10-01,G01,아낌e보금자리론,25800,20,신혼부부,반려
1,HF202610-00626,2026-10-01,B01,아낌e보금자리론,33000,15,생애최초,승인
2,HF202610-00475,2026-10-01,G02,아낌e보금자리론,22300,10,신혼부부,보류
3,HF202610-00482,2026-10-01,J02,보금자리론,41700,10,일반,반려
4,HF202610-00602,2026-10-01,C01,적격대출,12800,10,신혼부부,승인


## 4. 찾기

`지점코드` 열에서 `X99` 를 찾습니다. 결과는 행 번호 목록입니다.

In [5]:
found = s.search("X99", "지점코드")
print("찾은 행:", len(found), "건")
show(s, found)

찾은 행: 13 건


,신청번호,신청일,지점코드,상품명,신청금액(만원),대출기간(년),고객구분,처리상태
20,HF202610-00555,2026-10-03,X99,아낌e보금자리론,18900,10,일반,심사중
32,HF202610-00444,2026-10-05,X99,아낌e보금자리론,6600,40,생애최초,승인
43,HF202610-00431,2026-10-06,X99,아낌e보금자리론,24900,15,일반,보류
46,HF202610-00435,2026-10-06,X99,디딤돌대출,6200,20,생애최초,승인
79,HF202610-00546,2026-10-09,X99,적격대출,31500,20,다자녀,보류


## 5. 일괄 바꾸기

찾은 13건의 `X99` 를 `ON1` 로 한 번에 바꿉니다. 아직 **파일은 바뀌지 않았고** 메모리 안에서만 바뀐 상태입니다.

In [6]:
n = s.replace("지점코드", "X99", "ON1")
print(n, "칸을 바꿨습니다")
show(s, found)

13 칸을 바꿨습니다


,신청번호,신청일,지점코드,상품명,신청금액(만원),대출기간(년),고객구분,처리상태
20,HF202610-00555,2026-10-03,ON1,아낌e보금자리론,18900,10,일반,심사중
32,HF202610-00444,2026-10-05,ON1,아낌e보금자리론,6600,40,생애최초,승인
43,HF202610-00431,2026-10-06,ON1,아낌e보금자리론,24900,15,일반,보류
46,HF202610-00435,2026-10-06,ON1,디딤돌대출,6200,20,생애최초,승인
79,HF202610-00546,2026-10-09,ON1,적격대출,31500,20,다자녀,보류


## 6. 칸 하나 고치기 · 잘못된 값은 거부

숫자 열에 글자를 넣으면 프로그램이 받아 주지 않습니다.

In [7]:
rid = found[0]
s.set_cell(rid, "처리상태", "검토완료")
s.set_cell(rid, "신청금액(만원)", "19,000")        # 콤마가 있어도 숫자로 알아듣는다

try:
    s.set_cell(rid, "신청금액(만원)", "많이")
except ValueError as e:
    print("거부됨 →", e)

show(s, [rid])

거부됨 → [신청금액(만원)] 열은 숫자만 넣을 수 있습니다: '많이'


,신청번호,신청일,지점코드,상품명,신청금액(만원),대출기간(년),고객구분,처리상태
20,HF202610-00555,2026-10-03,ON1,아낌e보금자리론,19000,10,일반,검토완료


## 7. 행 추가 · 삭제 · 되돌리기

In [8]:
new = s.add_row({"신청번호": "HF202610-99999", "신청일": "2026-10-31", "지점코드": "ON1",
                 "상품명": "보금자리론", "신청금액(만원)": "12000", "대출기간(년)": "30",
                 "고객구분": "일반", "처리상태": "심사중"})
print("추가 후:", len(s.rows), "행")

s.delete_rows([found[1], found[2]])
print("삭제 후:", len(s.rows), "행")

s.undo()                                             # 마지막 작업(삭제)을 되돌린다
print("되돌린 뒤:", len(s.rows), "행")

추가 후: 241 행
삭제 후: 239 행
되돌린 뒤: 241 행


## 8. 저장하기 전에 — 무엇이 바뀌었나

저장하면 반영될 내용을 미리 봅니다.

In [9]:
lines = s.describe_changes()
print("바뀐 곳:", len(lines), "건\n")
for line in lines[:6]:
    print(line)
print("...")

바뀐 곳: 16 건

수정  HF202610-00555  [지점코드]  X99 → ON1
수정  HF202610-00555  [신청금액(만원)]  18900 → 19000
수정  HF202610-00555  [처리상태]  심사중 → 검토완료
수정  HF202610-00444  [지점코드]  X99 → ON1
수정  HF202610-00431  [지점코드]  X99 → ON1
수정  HF202610-00435  [지점코드]  X99 → ON1
...


## 9. 저장

원본 옆 `백업` 폴더에 복사본을 먼저 만든 뒤 저장합니다. 다시 읽어서 값이 들어갔는지 확인합니다.

In [10]:
backup = src.save("신청내역", s)
print("백업 파일:", backup.parent.name + "/" + backup.name[:len("보금자리론_신청_2026_10_")] + "날짜_시각.xlsx")

again = src.load("신청내역")
print("다시 읽은 행 수 :", len(again.rows))
print("ON1 인 행       :", len(again.search("ON1", "지점코드")))
print("X99 인 행       :", len(again.search("X99", "지점코드")))

백업 파일: 백업/보금자리론_신청_2026_10_날짜_시각.xlsx
다시 읽은 행 수 : 241
ON1 인 행       : 14
X99 인 행       : 0


## 10. pandas 로도 확인

같은 파일을 pandas 로 읽어 지점코드별 건수를 셉니다. 프로그램이 저장한 것은 보통의 엑셀 파일입니다.

In [11]:
df = pd.read_excel("samples/보금자리론_신청_2026_10.xlsx")
df["지점코드"].value_counts().head(5)

지점코드
J01    22
G02    21
S02    21
G01    18
B01    18
Name: count, dtype: int64

## 11. Access 테이블도 같은 방법으로

Access 는 `pyodbc` 와 Access 드라이버가 있는 PC 에서만 됩니다. 없으면 안내 문구만 나오고 넘어갑니다.
Access 는 어느 행을 고칠지 알려 주는 **기준 열**(값이 행마다 다른 열)이 필요합니다. `find_key` 가 찾아 줍니다.

In [12]:
try:
    db = ed.open_source("samples/업무DB.accdb")
    table = db.tables()[0]
    a = db.load(table)
    key = ed.find_key(a)
    print("테이블:", table, "· 행 수:", len(a.rows), "· 기준 열:", key)

    rejected = a.search("반려", "처리상태")
    n = a.fill("처리상태", "재심사", rejected)        # 반려 건을 모두 '재심사'로
    db.save(table, a, key=key)
    print(n, "건을 재심사로 바꿔 저장했습니다")
    print("다시 읽어 확인:", len(db.load(table).search("재심사", "처리상태")), "건")
except RuntimeError as e:
    print("이 PC 에서는 Access 를 쓸 수 없습니다.\n")
    print(e)

테이블: 신청내역 · 행 수: 240 · 기준 열: 신청번호


39 건을 재심사로 바꿔 저장했습니다


다시 읽어 확인: 39 건


## 12. 화면 프로그램으로 열어 보기

지금까지 한 일을 화면에서 하려면 VS Code 터미널에 아래처럼 입력합니다.

```
python data_editor.py samples\보금자리론_신청_2026_10.xlsx
```

## 연습 문제

1. `상품명` 열에서 `적격대출` 인 행은 몇 건인가요? (`s.search`)
2. `고객구분` 이 `다자녀` 인 행의 `처리상태` 를 모두 `우선심사` 로 바꿔 보세요. (`s.search` → `s.fill`)
3. 2번 칸을 다시 실행해 데이터를 처음으로 돌린 뒤, `지점코드표.xlsx` 를 열어 지점을 하나 추가하고 저장해 보세요.